In [ ]:
import h5py
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
import warnings
import vbn_utils
import decoding_utils as du
import ccf_utils
from vbn_utils import formatFigure
%matplotlib inline

In [ ]:
from notebook_utils import (
    calc_dprime,
    time_to_threshold_from_baseline_back_from_peak,
    conf_interval
)

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv" #"/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.5.0/project_metadata/ecephys_sessions.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'] = units['cortical_layer'].replace('3-Feb','2/3') # 2/3 sometimes gets incorrectly reformatted as a date

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

active_tensor = h5py.File(active_tensor_file)

sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/ccf_structure_tree_2017.csv")

In [ ]:
g_images = ['omitted'] + list(np.sort(stim_table[(stim_table['stimulus_name'].str.contains('_G_'))&
                    (~stim_table['omitted'])&
                    (~stim_table['image_name'].isin(['im083_r','im111_r']))]['image_name'].unique())) + ['im083_r','im111_r']

h_images = ['omitted'] + list(np.sort(stim_table[(stim_table['stimulus_name'].str.contains('_H_'))&
                    (~stim_table['omitted'])&
                    (~stim_table['image_name'].isin(['im083_r','im111_r']))]['image_name'].unique())) + ['im083_r','im111_r']

In [ ]:
image_dict = {'G': g_images, 'H': h_images}

In [ ]:
plt.rcParams['font.size'] = 14
warnings.filterwarnings("ignore")

## Calculate proportion of cells responsive over time for each area

### Compute (if pre-computed, load below)

In [ ]:
unit_ids = vbn_utils.get_unit_ids(units, ['VISall', 'SCMRN', 'LGd', 'LP'])
session_list = units.set_index('unit_id').loc[unit_ids]['ecephys_session_id'].unique()

session_responsiveness_over_time = vbn_utils.unit_responsiveness_over_time(active_tensor_file, session_list, unit_ids)

In [ ]:
rot_dict = {}
for srot in session_responsiveness_over_time:
    unitid = srot['unit_ids'][0]
    session_id = units.set_index('unit_id').loc[unitid]['ecephys_session_id']
    rot_dict[session_id] = srot

In [ ]:
import pickle

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/responsiveness_over_time_dict.pkl", 'wb') as file:
    pickle.dump(rot_dict, file)

### Load if pre-computed

In [ ]:
import pickle

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/responsiveness_over_time_dict.pkl", 'rb') as file:
    rot_dict = pickle.load(file)

In [ ]:
no_abnorm = sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()
novel_sessions = sessions_table[(sessions_table['experience_level']=='Novel')&no_abnorm]['ecephys_session_id'].values
familiar_sessions = sessions_table[(sessions_table['experience_level']=='Familiar')&no_abnorm]['ecephys_session_id'].values

### Plot responsiveness over time

In [ ]:
region_names = ['LGd', 'VISp', 'VISl', 'VISrl', 'VISal', 'LP', 'VISpm', 'VISam', 'Hipp', 'SCMRN', 'VISall']

cell_type = 'all'
plt.rcParams['font.size'] = 16
region_summary_rot = {r:{'Novel_unshared':[], 'Novel_shared':[], 'Familiar_unshared':[], 'Familiar_shared':[]} for r in region_names}
for region in region_names:
    for sessions, condition in zip([novel_sessions, familiar_sessions], ['Novel', 'Familiar']):
        for session_id in sessions:
            
            session_rot = rot_dict[session_id]
            
            units_to_analyze = vbn_utils.get_unit_ids(units, region, cell_types=cell_type, session_id=session_id, 
                                                            training_trajectory='all')
            units_to_analyze = units[units['unit_id'].isin(units_to_analyze)]
            
            if len(units_to_analyze)==0:
                continue
            
            response_inds = np.isin(session_rot['unit_ids'], units_to_analyze['unit_id'].values)

            shared_images = ['im083_r', 'im111_r']
            non_shared_images = [k for k,v in session_rot.items() if 'im' in k and ~np.isin(k, shared_images)]

            for image_type_set, image_label in zip([shared_images, non_shared_images], ['_shared', '_unshared']):
                image_type_data = []
                for im in image_type_set:
                    im_sparseness = session_rot[im][response_inds]
                    sig = im_sparseness<0.01
                    image_type_data.append(np.mean(sig, axis=0))
                
                region_summary_rot[region][f'{condition}{image_label}'].append(np.mean(image_type_data, axis=0))


    fig, axes = plt.subplots(1,2)
    fig.set_size_inches([8,4.5])
    ax = axes[0]
    time = np.arange(100) + 20
    color_dict = {'Familiar_unshared': 'b', 'Novel_shared': 'purple', 'Novel_unshared': 'r'}
    for key in ['Familiar_unshared','Novel_unshared']:
        num_no_nan = np.sum([np.all(~np.isnan(r)) for r in region_summary_rot[region][key]])

        mean = np.nanmean(region_summary_rot[region][key], axis=0)[:100]
        sem = np.nanstd(region_summary_rot[region][key], axis=0)/num_no_nan**0.5
        sem = sem[:100]
        ax.plot(time, mean, color=color_dict[key], label=key)
        ax.fill_between(time, mean+sem, mean-sem, color=color_dict[key], alpha=0.5)

        vals = np.stack(region_summary_rot[region][key])
        for itimepoint, timepoint in enumerate([20, 60]):
            time_mean = np.nanmean(vals[:, timepoint])
            time_sem = np.nanstd(vals[:, timepoint])/num_no_nan**0.5
            axes[1].plot(itimepoint, time_mean, color=color_dict[key], marker='o', markersize=10)
            axes[1].errorbar(itimepoint, time_mean, yerr=time_sem, color=color_dict[key], capsize=5)
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels(['20-60ms', '60-100ms'])   
    
    ax.set_title(region)
    ax.set_xlabel('Time from stim start (ms)')
    ax.set_ylabel('Fraction cells responsive')
    ymax = ax.get_ylim()[1]
    [ax.plot(xval, ymax, 'kv', mfc=mfc, ms=10) for mfc, xval in zip(['k', 'w'], [40, 80])]
    
    ax.set_ylim([0, ax.get_ylim()[1]])
    ax.set_xticks((20, 100))
    ax.spines['bottom'].set_bounds(20, 100)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)  # Set spine linewidth to 1.5 points

    # Set the linewidth of ticks
    ax.tick_params(width=1.5) 
    [vbn_utils.formatFigure(fig, a) for a in axes]
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5)) # 5 ticks on the y-axis
    ax.xaxis.set_label_coords(0.4, -0.12)
    plt.tight_layout()

In [ ]:
from matplotlib.lines import Line2D

plt.rcParams['font.size'] = 18

fig, ax = plt.subplots()
regions = ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
for ir, region in enumerate(regions):
    
    iteration_timepoint_vals = np.full((2, 2, 1000), np.nan)
    for ikey, key in enumerate(['Familiar_unshared', 'Novel_unshared']):
        vals = np.stack(region_summary_rot[region][key])

        for iteration in range(1000):
            iteration_indices = np.random.choice(np.arange(vals.shape[0]), vals.shape[0], replace=True)
            for itimepoint, timepoint in enumerate([20, 60]):
                time_mean = np.nanmean(vals[iteration_indices, timepoint])
                iteration_timepoint_vals[ikey, itimepoint, iteration] = time_mean
        
    
    iteration_diffs = iteration_timepoint_vals[1] - iteration_timepoint_vals[0]
    iteration_means = iteration_diffs.mean(axis=1)

    ci_high = np.percentile(iteration_diffs, 97.5, axis=1)
    ci_low = np.percentile(iteration_diffs, 2.5, axis=1)

    color = ccf_utils.get_area_color(region, structure_tree)
    for mfc, ind in zip([color, 'w'], [0,1]):
        ax.plot(ir, iteration_means[ind], 'o', color=color, mfc=mfc, ms=11, markeredgewidth=1.5)
        ax.errorbar([ir], [iteration_means[ind]], yerr=[[ci_high[ind]-iteration_means[ind]], [iteration_means[ind]-ci_low[ind]]], 
                        color=color)

ax.axhline(0, color='k', ls='dotted')
ax.set_xticks(np.arange(len(regions)))
ax.set_xticklabels(regions, rotation=90)
ax.set_ylabel(r"$\Delta$ fraction responsive (Nov - Fam)")
for spine in ax.spines.values():
    spine.set_linewidth(1.5)  # Set spine linewidth to 1.5 points

# Set the linewidth of ticks
ax.tick_params(width=1.5) 

custom_lines = [
    Line2D([0], [0], marker='o', color='k', markerfacecolor='w', markersize=10, lw=0, label='late'),
    Line2D([0], [0], marker='o', color='k', markerfacecolor='k', markersize=10, lw=0, label='early'),
]

plt.legend(handles=custom_lines, loc='upper left', frameon=False, bbox_to_anchor=(-0.05, 1.1), handletextpad=0)
vbn_utils.formatFigure(fig, ax)

## Load precomputed PSTH data for each unit (created in Figure 4 notebook)

In [ ]:
import pickle

with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/change_prechange_responses_active_passive_by_unitid.pkl", 'rb') as file:
    flash_data = pickle.load(file)

## Calculate visual latency and novelty modulation latency for each area

### Compute latencies (if pre-computed, load below)

In [ ]:
from analysis_utils import exponential_convolve

base_slice = slice(0, 50)
base_sub = lambda x: x - x[:, base_slice].mean(axis=1)[:, None]

regions = ['LGd',
    'LP',
    'VISp',
    'VISl',
    'VISal',
    'VISrl',
    'VISpm',
    'VISam',
    'SCMRN',
    'VISall'
    ]

clusters_to_plot = 'all' 

conditions = ['Familiar', 'Novel', 'Familiar_omitted', 'Novel_omitted']

baseline_samples = 50

latencies = {r:{c:{'Familiar': [], 'Novel': [], 'dprime': []} for c in ['all', 'RS', 'FS', 'SST']} for r in regions}
for cell_type in ['all', 'RS', 'FS', 'SST']:
    for region in regions:
        if cell_type != 'all' and 'VIS' not in region:
            continue

        data_dict = {}
        for condition in conditions:
            if 'omitted' in condition:
                image_category = 'omission'
            else:
                image_category = 'nonshared_nonchange'

            units_to_use = vbn_utils.get_unit_ids(units, region, cell_types=cell_type, layers='all', clusters='all', clustering = 'new', 
                    experience=condition.split('_')[0], responsive=False, session_id='all')
            
            data = [flash_data[u]['active'][image_category] for u in units_to_use]
            
            data = np.array([exponential_convolve(d, 3, symmetrical=True) for d in data])
            data_dict[condition] = base_sub(data)
        
        fam = data_dict['Familiar']
        nov = data_dict['Novel']
        fam_omitted = data_dict['Familiar_omitted']
        nov_omitted = data_dict['Novel_omitted']
        
        num_fam = fam.shape[0]
        num_nov = nov.shape[0]

        dprime_iterations = []
        dprime_null_iterations = []
        diff_iterations = []
        time_to_thresh = []
        for iteration in range(1000):
            dprime_over_time = []
            dprime_fvso_over_time = []
            dprime_nvso_over_time = []

            sample_fam = fam[np.random.choice(np.arange(num_fam), num_fam, replace=True)]
            sample_nov = nov[np.random.choice(np.arange(num_nov), num_nov, replace=True)]
            sample_fam_omitted = fam_omitted[np.random.choice(np.arange(num_fam), num_fam, replace=True)]
            sample_nov_omitted = nov_omitted[np.random.choice(np.arange(num_nov), num_nov, replace=True)]
            

            for timepoint in range(fam.shape[1]):
                dp = calc_dprime(sample_nov[:, timepoint], sample_fam[:, timepoint], signed=True)
                dprime_over_time.append(dp)

                dp_fvso = calc_dprime(sample_fam[:, timepoint], sample_fam_omitted[:, timepoint], signed=True)
                dp_nvso = calc_dprime(sample_nov[:, timepoint], sample_nov_omitted[:, timepoint], signed=True)
                dprime_fvso_over_time.append(dp_fvso)
                dprime_nvso_over_time.append(dp_nvso)


            latencies[region][cell_type]['dprime'].append(time_to_threshold_from_baseline_back_from_peak(dprime_over_time[baseline_samples:], threshold=0.1)[0] + baseline_samples)
            latencies[region][cell_type]['Familiar'].append(time_to_threshold_from_baseline_back_from_peak(dprime_fvso_over_time[baseline_samples:], threshold=0.1)[0] + baseline_samples)
            latencies[region][cell_type]['Novel'].append(time_to_threshold_from_baseline_back_from_peak(dprime_nvso_over_time[baseline_samples:], threshold=0.1)[0] + baseline_samples)

            dprime_iterations.append(dprime_over_time)

        median_dprime_over_samples = np.median(dprime_iterations, axis=0)
        cis = np.array([conf_interval(vals) for vals in np.array(dprime_iterations).T])

        fam_mean = np.nanmean(fam, axis=0)
        nov_mean = np.nanmean(nov, axis=0)

        median_dprime_time_to_thresh = np.nanmedian(latencies[region][cell_type]['dprime'])
        fig, ax = plt.subplots()
        fig.suptitle(f'{region} {cell_type} \n fam n: {num_fam}, nov n: {num_nov}')
        ax2 = ax.twinx()
        ax.plot(median_dprime_over_samples, 'k')
        ax.plot(median_dprime_time_to_thresh, median_dprime_over_samples[int(median_dprime_time_to_thresh)], 'ko')
        ax.fill_between(np.arange(len(median_dprime_over_samples)), cis[:,0], cis[:,1], color='k', alpha=0.3, lw=0)
        ax.axvline(np.nanmedian(latencies[region][cell_type]['dprime']), color='k', ls='dotted')

        ax2.plot(np.nanmean(fam, axis=0), 'b')
        ax2.plot(np.nanmean(nov, axis=0), 'r')
        ax2.plot(np.nanmean(fam_omitted, axis=0), 'b', alpha=0.5)
        ax2.plot(np.nanmean(nov_omitted, axis=0), 'r', alpha=0.5)
        ax2.axvline(np.nanmedian(latencies[region][cell_type]['Familiar']), color='b', ls='dotted')
        ax2.axvline(np.nanmedian(latencies[region][cell_type]['Novel']), color='r', ls='dotted')

    

In [ ]:
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/latencies_by_region_and_celltype.pkl", 'wb') as file:
    pickle.dump(latencies, file)

### Load pre-computed latencies

In [ ]:
# Load latencies for plotting
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/latencies_by_region_and_celltype.pkl", 'rb') as file:
    latencies = pickle.load(file)

### Plot visual and novelty modulation latencies vs. anatomical hierarchy

In [ ]:
import scipy.stats

plt.rcParams.update({'font.size':14.5})

hierarchy_dict ={
                    'LGd':	-0.515027963,
                    'VISp':	-0.357332099,
                    'VISl':	-0.093888551,
                    'VISrl':-0.059871325,
                    'LP':	0.10524781,
                    'VISal':0.152217979,
                    'VISpm':0.327668075,
                    'VISam':0.440986074
                }

fig, axes = plt.subplots(2,1)
fig.set_size_inches(5,10)
ax = axes[0]
ax2 = axes[1]

binsize = 1
vis_latencies = []
dprime_latencies = []
hierarchy_positions = []
for ir, region in enumerate(['LGd', 'VISp', 'VISl', 'VISrl', 'VISal', 'LP', 'VISpm', 'VISam']):
        visual_latency_array = np.stack([latencies[region]['all'][cond] for cond in ['Familiar', 'Novel']])
        visual_latency_mean_over_cond = np.mean(visual_latency_array, axis = 0) - 50
        visual_latency_mean_over_cond[(visual_latency_mean_over_cond>270)|(visual_latency_mean_over_cond<20)] = np.nan
        vis_latencies.append(visual_latency_mean_over_cond)
        dprime_latency = np.array(latencies[region]['all']['dprime']) - 50
        dprime_latency[(dprime_latency>270)|(dprime_latency<20)] = np.nan
        dprime_latencies.append(dprime_latency)
        hierarchy_positions.append(hierarchy_dict[region])
        ax.plot(hierarchy_dict[region], np.nanmean(visual_latency_mean_over_cond), 'o', color=ccf_utils.get_area_color(region, structure_tree), ms=10)
        ax.errorbar(hierarchy_dict[region], np.nanmean(visual_latency_mean_over_cond), np.nanstd(visual_latency_mean_over_cond), color=ccf_utils.get_area_color(region, structure_tree), ls='none')
        
        ax2.plot(hierarchy_dict[region], np.nanmean(dprime_latency), 'o', color=ccf_utils.get_area_color(region, structure_tree), ms=10)
        ax2.errorbar(hierarchy_dict[region], np.nanmean(dprime_latency), np.nanstd(dprime_latency), color=ccf_utils.get_area_color(region, structure_tree), ls='none',)

ax.set_ylabel('Visual Response Latency (ms)')
ax2.set_ylabel('Novelty Modulation Latency (ms)')

xlims = np.array(ax.get_xlim())
corr_params = scipy.stats.linregress(hierarchy_positions, np.nanmean(vis_latencies, axis=1))
ax.plot(xlims, xlims*corr_params[0] + corr_params[1], 'k--',)

ax.set_xticks([val for key, val in hierarchy_dict.items()])
ax.set_xticklabels([f'{np.round(val, 2)} \n{key}' if key not in ['VISrl', 'VISal', 'VISam'] else f'\n\n{key}' for key, val in hierarchy_dict.items()])

[a.set_title('') for a in [ax, ax2]]
for corr_func in [scipy.stats.linregress, scipy.stats.spearmanr]:
        vis_response_corr = corr_func(hierarchy_positions, np.nanmean(vis_latencies, axis=1))
        dprime_corr = corr_func(hierarchy_positions, np.nanmean(dprime_latencies, axis=1))

        print(f'Vis latency {corr_func.__name__}: {vis_response_corr}')
        print(f'Dprime latency {corr_func.__name__}: {dprime_corr}')
        atitle = ax.get_title()
        ax.set_title(atitle + '\n' + corr_func.__name__ + ' pval:' + str(vis_response_corr.pvalue))

        atitle = ax2.get_title()

ax2.set_xticks([val for key, val in hierarchy_dict.items()])
ax2.set_xticklabels([f'{np.round(val, 2)} \n{key}' if key not in ['VISrl', 'VISal', 'VISam'] else f'\n\n{key}' for key, val in hierarchy_dict.items()])

[formatFigure(fig, a) for a in axes]
plt.tight_layout()

## Novelty modulation PSTHs for LGd and VIS

In [ ]:
areas_ordered = ['LGd', 'VISall']


for cell_type in ['RS', 'FS', 'SST', 'all']:
    fig = vbn_utils.make_nov_mod_psth_figure(areas_ordered, layers='all', cell_types=cell_type, clusters='all', units_subsample=units, flash_data=flash_data, norm=False, state=['active',],
                        flashes=['nonshared_nonchange', 'change'], training_trajectory='all', plot_pvalues=False, smoothing_kernel_width=3, return_fig=True)
    axes = fig.get_axes()

    for area, ax in zip(areas_ordered, [axes[0], axes[1]]):
        if cell_type != 'all' and 'VIS' not in area:
            continue
        visual_latency_array = np.stack([latencies[area][cell_type][cond] for cond in ['Familiar', 'Novel']])
        visual_latency_mean_over_cond = np.mean(visual_latency_array, axis = 0) - 50
        visual_latency_mean_over_cond[(visual_latency_mean_over_cond>270)|(visual_latency_mean_over_cond<20)] = np.nan

        dprime_latency = np.array(latencies[area][cell_type]['dprime']) - 50
        dprime_latency[(dprime_latency>270)|(dprime_latency<20)] = np.nan

        ylims = ax.get_ylim()
        ax.plot(np.nanmean(visual_latency_mean_over_cond), ylims[1], 'kv', mfc='w', ms=10)
        ax.plot(np.nanmean(dprime_latency), ylims[1], 'kv', ms=10)

        ax.spines['bottom'].set_bounds(20, 100)
        for spine in ax.spines.values():
            spine.set_linewidth(1.5)  # Set spine linewidth to 1.5 points



## Image-wise response metrics (tuning changes Familiar→Novel)

### Compute image-wise responses (if pre-computed, load from units table)

In [ ]:
unit_ids = vbn_utils.get_unit_ids(units, ['VISall', 'SCMRN', 'LGd', 'LP'])
session_list = units.set_index('unit_id').loc[unit_ids]['ecephys_session_id'].unique()

session_unit_imagewise_responses = vbn_utils.unit_imagewise_stats(active_tensor_file, session_list, unit_ids, baseline_slice=slice(700, 750), response_slice=slice(20,100))

In [ ]:
unit_imagewise_responses = {}
for d in session_unit_imagewise_responses:
    unit_imagewise_responses.update(d)

In [ ]:
ggh = units.loc[du.apply_condition_filter(units, 'Familiar', 'GGH') | du.apply_condition_filter(units, 'Novel', 'GGH')]
hhg = units.loc[du.apply_condition_filter(units, 'Familiar', 'HHG') | du.apply_condition_filter(units, 'Novel', 'HHG')]
ghg = units.loc[du.apply_condition_filter(units, 'Familiar', 'GHG') | du.apply_condition_filter(units, 'Novel', 'GHG')]

### Tuning curve analysis

In [ ]:
vis_units = vbn_utils.get_unit_ids(ggh, 'VISall')

In [ ]:
image_ids = h_images[:-2] + g_images[1:]
plt.rcParams['font.size'] = 22

#tuning curves
colors = ['b', 'r']
regions = ['LGd', 'LP', 'VISall', 'VISp', 'VISl', 'VISal', 'VISrl', 'VISpm', 'VISam']
tuning_diff_dict = {}
for cell_type in ['RS', 'FS']:
    for region in regions:
        
        condition_curves = []
        fig, ax = plt.subplots()
        ax2 = ax.twinx()
        for ic, condition in enumerate(['Familiar', 'Novel']):

            units_to_use = vbn_utils.get_unit_ids(units, region, experience=condition, cell_types=cell_type)

            tuning_curves = []
            for u in units_to_use:
                tuning = np.array([unit_imagewise_responses[u][im]['mean'] for im in image_ids[1:-2]])
                tuning = tuning[~np.isnan(tuning)]
                if len(tuning)<6:
                    continue

                tuning_curves.append(np.sort(tuning)*1000)

            tuning_curves = np.array(tuning_curves)

            iteration_curves = []
            for iteration in range(1000):
                indices = np.random.choice(np.arange(tuning_curves.shape[0]), tuning_curves.shape[0], replace=True)
                iteration_curves.append(np.nanmean(tuning_curves[indices], axis=0))

            condition_curves.append(iteration_curves)

            vbn_utils.mean_sem_plot(tuning_curves, ax, x=np.arange(1, 7), color=colors[ic])
        
        ax.set_title(f'{region} {cell_type}')
        
        condition_curves = np.array(condition_curves)
        
        vbn_utils.mean_CI_plot(condition_curves[1]-condition_curves[0], ax2, x=np.arange(1,7), color='k', ls='dotted')
        tuning_diff_dict[(region, cell_type)] = condition_curves[1] - condition_curves[0]
        
        ax.set_zorder(ax2.get_zorder()+1)
        ax.patch.set_visible(False)

        ax.set_xticks(np.arange(1, 7))
        ax.set_xlabel('Image rank')
        ax.set_ylabel('Firing rate (Hz)')
        ax2.set_ylabel('Difference in firing rate (Hz)', rotation=270, labelpad=23)
        
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))  # 5 ticks on the y-axis
        ax2.yaxis.set_major_locator(MaxNLocator(nbins=5))  # 5 ticks on the y-axis


        for a in [ax, ax2]:
            for spine in a.spines.values():
                spine.set_linewidth(1.25)  # Set spine linewidth to 2 points

            # Set the linewidth of ticks
            a.tick_params(width=1.25)

        [vbn_utils.formatFigure(fig, a, yaxis_side=side) for a,side in zip([ax, ax2], ['left', 'right'])]
